# Python for Data Engineering

- **Python `venv` + activation:** [Virtual Environments and Packages](https://docs.python.org/3/tutorial/venv.html) · [`venv` module](https://docs.python.org/3/library/venv.html)
- **Install Jupyter tools:** [Project Jupyter — Installing Jupyter](https://jupyter.org/install)
- **JupyterLab + `pip` notes (e.g. `PATH` if you use `--user`):** [JupyterLab — Installation](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html)

### Why Jupyter for this course?

- **Mix explanation and code:** Markdown cells for notes, code cells for runnable snippets—good for teaching **pipelines** step by step.
- **See outputs immediately:** Run small **transforms** (like cleaning lists) and inspect **stdout** without switching tools.
- **Industry habit:** **Notebooks** are a common way to **explore** data and share a **reproducible** story before hardening logic in jobs or services.
- **JupyterLab** (recommended on [jupyter.org/install](https://jupyter.org/install)) is the **browser-based** app you start with `jupyter lab`; it is the current default “full” Jupyter UI for labs and teams.

### Step 0 — Install Python (if you do not have it)

- **All OS:** Install a recent **Python 3** (JupyterLab 4 supports roughly **3.9–3.13** per [JupyterLab installation docs](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html)) from [python.org](https://www.python.org/downloads/) or your OS/vendor instructions.
- **Reason:** `python` / `pip` / `venv` below assume a working interpreter on your `PATH`.

### Step 1 — Create and activate a virtual environment (`venv`)

**Reason:** Isolates **`pip` packages** (Jupyter, etc.) for this class so they do not clash with other projects—see [Python venv tutorial](https://docs.python.org/3/tutorial/venv.html).

**Create** (run **one** block for your OS; folder name `.venv` is a common convention from the same tutorial):

```bash
python3 -m venv .venv
```

On Windows, if `python3` is not on `PATH`, use:

```text
python -m venv .venv
```

**Activate** (must run in **every new terminal** before `pip install` / `jupyter lab`):

| OS / shell | Exact command | Why |
|--------------|---------------|-----|
| **macOS / Linux** (`bash` / `zsh`) | `source .venv/bin/activate` | Puts this env’s `python`/`pip` first on `PATH` ([docs](https://docs.python.org/3/tutorial/venv.html)). |
| **Windows** (Command Prompt) | `.venv\Scripts\activate.bat` | Same idea; Windows uses `Scripts` instead of `bin`. |
| **Windows** (PowerShell) | `.venv\Scripts\Activate.ps1` | If **execution policy** blocks the script, see Microsoft’s [about_Execution_Policies](https://learn.microsoft.com/powershell/module/microsoft.powershell.core/about/about_execution_policies) (often `Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser` for development). |

**Optional (Debian/Ubuntu):** if `python3 -m venv` fails, install the `venv` component for your distro (e.g. `python3-venv`)—see your distro’s Python packaging docs.

**Deactivate** when finished (any OS): `deactivate`

### Step 2 — Upgrade `pip` and install JupyterLab + `ipykernel`

**Reason:** **`pip`** is the standard installer ([Jupyter recommends `pip`](https://jupyter.org/install)). **`ipykernel`** lets **VS Code / Cursor** run `.ipynb` files against this same environment (otherwise the editor may not find a notebook kernel).

```bash
python -m pip install --upgrade pip
pip install jupyterlab ipykernel
```

Use `python3 -m pip` instead of `python -m pip` on macOS/Linux if that is what your shell uses after activation.

**If you used `pip install --user`:** on macOS/Linux you may need `jupyter` on your `PATH` ([JupyterLab docs](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html)):

```bash
export PATH="$HOME/.local/bin:$PATH"
```

### Step 3 — Start JupyterLab

From the **activated** environment ([jupyter.org/install](https://jupyter.org/install)):

```bash
jupyter lab
```

**Reason:** Starts the **Jupyter server** and opens the **JupyterLab** UI (file browser, notebooks, terminals) in your **browser**—see [JupyterLab installation overview](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html).

**Optional (macOS / Linux with Homebrew):** [jupyter.org/install](https://jupyter.org/install) also documents `brew install jupyterlab` if you prefer Homebrew’s packaging on those platforms.

### Step 4 — Open this notebook

- **In JupyterLab:** navigate to this `.ipynb` and open it.
- **In Cursor / VS Code:** open the folder, select the **`.venv` Python interpreter**, then open this notebook so the kernel uses the env where you installed **`ipykernel`**.

**In the notebook**, run the next cells to confirm **`python`** and **`pip`** match the environment you think you are using.

In [1]:
# Environment check — keywords: import, sys, platform
import sys
import platform

print("python_version:", platform.python_version())
print("executable:", sys.executable)
print("ok_for_class:", sys.version_info >= (3, 9))

python_version: 3.13.7
executable: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13
ok_for_class: True


In [2]:
# pip version inside this kernel (Jupyter line magic — keyword: %pip)
%pip --version

pip 26.0.1 from /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip (python 3.13)
Note: you may need to restart the kernel to use updated packages.


## Keywords cheat-sheet (Day 1)

| Keyword | Meaning (short) |
|--------|------------------|
| **Data Engineering** | Build **pipelines**: move + shape data for analytics/apps |
| **ingest** / **extract** | Read **raw** data from source |
| **raw** | As landed; may be **dirty** |
| **validate** / **clean** | Drop or fix bad rows |
| **`list`** | Ordered collection `[...]` |
| **`dict`** | Key-value record `{"key": value}` |
| **`for`** | Loop over each item |
| **`if`** | Branch on a condition |
| **`.append()`** | Add one element to a **list** |
| **`print()`** | Send text to **stdout** (see output while developing) |
| **ETL** | **E**xtract → **T**ransform → **L**oad |
| **Spark** | Distributed engine for **big** data (later)

## Context (short)

- **Data Engineering:** **pipelines** from sources → **warehouse** / lake → consumers.  
- **Python:** scripting **ETL**, calling **APIs**, SQL drivers, **Spark** jobs.  
- **Example domain:** **orders**, **users**, **payments** (e-commerce).

## Read **raw** data (`list` of `dict`)

**Keywords:** **raw** · **dirty** · **`list`** · **`dict`** · **record** / row

Below: fake **extract** — one bad **amount** on purpose (**invalid** for our rule later).

In [3]:
# Raw extract → Python structures (keywords: list, dict, keys, values)
orders = [
    {"id": 1, "amount": 250},
    {"id": 2, "amount": -100},
    {"id": 3, "amount": 450},
]

print("raw_orders:", orders)
print("len(raw):", len(orders))

raw_orders: [{'id': 1, 'amount': 250}, {'id': 2, 'amount': -100}, {'id': 3, 'amount': 450}]
len(raw): 3


## `print` and **control flow**

**Keywords:** **`print()`** · **statement** · runs **top to bottom** (control flow)

In [4]:
# Sequential execution — keyword: print
print("step_1_start")
total_before_cleaning = 250 + (-100) + 450
print("naive_sum_includes_bad_row:", total_before_cleaning)
print("step_3_end")

step_1_start
naive_sum_includes_bad_row: 600
step_3_end


## **Clean** with **`for`** + **`if`** + **`.append()`**

**Keywords:** **filter** · **validate** · **`for`** · **`if`** · **`>`** · **`.append()`** · **`clean_orders`**

**Rule:** keep only `order["amount"] > 0` (**drop** negatives).

In [5]:
# Transform: raw list → clean list (same pattern as tiny ETL "T" step)
clean_orders = []
for order in orders:
    if order["amount"] > 0:
        clean_orders.append(order)

print("clean_orders:", clean_orders)
print("len(raw):", len(orders), "len(clean):", len(clean_orders))

clean_orders: [{'id': 1, 'amount': 250}, {'id': 3, 'amount': 450}]
len(raw): 3 len(clean): 2


## What happened (**keywords**)

- **Input:** **`orders`** = **raw** / **dirty**  
- **Output:** **`clean_orders`** = **validated** subset  
- **Pipeline idea:** **raw → clean** (first **trust** step before **aggregate** / **load**)
- **Pattern:** **ETL** — here: **extract** (list above) → **transform** (loop) → **load** (not shown; would be file/API/warehouse)

## **Scale** (one idea)

**Keywords:** **volume** · **batch** · **distributed** · **Apache Spark**

Today: **3** rows. Production: **millions+** per **partition** / day — same **logic**, often **Spark** (or SQL in a **warehouse**) for **parallel** work.

## Exercise (**keywords:** extend **`list`**, new **`if`**, **edge case**)

1. **`append`** more **`dict`** rows to **`orders`** (valid + invalid).  
2. Add another **invalid** case (e.g. `amount: 0` or missing **`amount`** — use **`.get()`** or **`in`**).  
3. Change the **`if`** so the **filter** matches your **rule** (e.g. `>= 100`).

Edit the next **cell**; **`run`** top-to-bottom if **`NameError`** on **`orders`**.

In [ ]:
# Practice: my_orders, for, if, append, print
# my_orders = [...]
# my_clean = []
# for o in my_orders:
#     ...
# print(my_clean)

print("Edit this cell.")

---

### End of Day 1

**Takeaway keywords:** **raw → clean** · **`for`/`if`** · **`print`** for visibility · next: **functions**, **files**, **I/O**, then **Spark**/warehouse tools.